In [5]:
import boto3
import requests
import json
from datetime import datetime
from abc import ABC, abstractmethod

class BaseETL(ABC):
    
    def __init__(self, bucket='bronze'):
        self.s3 = boto3.client(
            's3',
            endpoint_url='http://storage:9000',
            aws_access_key_id='root',
            aws_secret_access_key='password',
            region_name='us-east-1'
        )
        self.bucket = bucket
        self.date_str = datetime.now().strftime("%Y/%m/%d")
    
    @abstractmethod
    def extract(self) -> list:
        print('You shouldnt use this method naked')
        pass
    
    def load(self, data: list, key: str):
        self.s3.put_object(
            Bucket=self.bucket,
            Key=key,
            Body=json.dumps(data, ensure_ascii=False),
            ContentType='application/json'
        )
        print(f"{len(data)} records → {self.bucket}/{key}")
    
    def run(self, key: str):
        print(f"{self.__class__.__name__} starting")
        data = self.extract()
        self.load(data, key)
        print(f"Done — {len(data)} records landed")
        return data